In [1]:
import os
import h5py
import requests
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive,userdata
drive.mount("/content/drive")
import scipy.stats as stats
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import time

Mounted at /content/drive


In [3]:
os.chdir("/content/drive/MyDrive/TNGdata2")

In [4]:
baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":userdata.get("TNG_API_KEY")}

In [5]:
def get(path, params=None):
    # make HTTP GET request to path
    r = requests.get(path, params=params, headers=headers)

    # raise exception if response code is not HTTP SUCCESS (200)
    r.raise_for_status()

    if r.headers['content-type'] == 'application/json':
        return r.json() # parse json responses automatically

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        return filename # return the filename string

    return r

In [6]:
r = get(baseUrl)

In [7]:
tng50d1 = get(baseUrl + "TNG50-1")
tng50d2 = get(baseUrl + "TNG50-2")
tng50d3 = get(baseUrl + "TNG50-3")

In [9]:
def get_and_rename(url, params, dest_name):
    temp_filename = get(url, params)
    if os.path.exists(temp_filename) and temp_filename != dest_name:
        if os.path.exists(dest_name):
            os.remove(dest_name)
        os.rename(temp_filename, dest_name)
    return dest_name

subhalo50d1_cutout = get_and_rename(tng50d1['url'] + "/snapshots/33/subhalos/0/cutout.hdf5", {"gas":"Coordinates,MagneticField"}, "cutout_50d1_0.hdf5")
subhalo50d2_cutout = get_and_rename(tng50d2['url'] + "/snapshots/33/subhalos/0/cutout.hdf5", {"gas":"Coordinates,MagneticField"}, "cutout_50d2_0.hdf5")
subhalo50d3_cutout = get_and_rename(tng50d3['url'] + "/snapshots/33/subhalos/0/cutout.hdf5", {"gas":"Coordinates,MagneticField"}, "cutout_50d3_0.hdf5")

In [10]:
cutouts_paths = {
    "1": subhalo50d1_cutout,
    "2": subhalo50d2_cutout,
    "3": subhalo50d3_cutout
}

mag_raw = {}
mag_physical_dict = {}
mag_physical_magnitude_dict = {}
coords_dict = {}
headers_dict = {}

for res in ["1", "2", "3"]:
    cutout_file = cutouts_paths[res]
    with h5py.File(cutout_file, 'r') as f:
        coords_dict[res] = f['PartType0']['Coordinates'][:]
        mag_raw[res] = f['PartType0']['MagneticField'][:]
        headers_dict[res] = dict(f['Header'].attrs)

    h_param = headers_dict[res]['HubbleParam']
    scale_f = 1.0 / (1.0 + headers_dict[res]["Redshift"])

    mag_physical_dict[res] = mag_raw[res] * h_param / scale_f**2 * 2.6e-6

    mag_physical_magnitude_dict[res] = np.sqrt(
        mag_physical_dict[res][:, 0]**2 +
        mag_physical_dict[res][:, 1]**2 +
        mag_physical_dict[res][:, 2]**2
    )

In [11]:
for res in ["1", "2", "3"]:
    m_norm = mag_physical_magnitude_dict[res]
    print(f"Resolution TNG50-{res}:")
    print(f"  Median: {np.median(m_norm):.5e}")
    print(f"  Mean:   {np.mean(m_norm):.5e}")
    print(f"  Min:    {np.min(m_norm):.5e}")
    print(f"  Max:    {np.max(m_norm):.5e}\n")

Resolution TNG50-1:
  Median: 1.15274e-06
  Mean:   5.78380e-06
  Min:    2.21321e-12
  Max:    7.93010e-03

Resolution TNG50-2:
  Median: 8.92205e-07
  Mean:   2.47200e-06
  Min:    1.17414e-12
  Max:    4.10281e-03

Resolution TNG50-3:
  Median: 5.25733e-07
  Mean:   1.60938e-06
  Min:    1.38649e-12
  Max:    7.59466e-05



In [ ]:
subhalo_data = {}
for res in ["1", "2", "3"]:
    sim_name = f"TNG50-{res}"
    subhalo_data[res] = get(baseUrl + sim_name + "/snapshots/33/subhalos/0")

sfrs_res0 = {res: subhalo_data[res]['sfr'] for res in ["1", "2", "3"]}
bhmdots_res0 = {res: subhalo_data[res]['bhmdot'] for res in ["1", "2", "3"]}
mstars_res0 = {res: subhalo_data[res]['mass_stars'] for res in ["1", "2", "3"]}
mbhs_res0 = {res: subhalo_data[res]['mass_bhs'] for res in ["1", "2", "3"]}

for res in ["1", "2", "3"]:
    print(f"Resolution TNG50-{res}: sfr={sfrs_res0[res]:.3f}, bhmdot={bhmdots_res0[res]:.3e}, mstar={mstars_res0[res]:.3f}, mbh={mbhs_res0[res]:.3f}")

Resolution TNG50-1: sfr=267.727, bhmdot=4.977e-04, mstar=36.256, mbh=0.040
Resolution TNG50-2: sfr=69.092, bhmdot=1.416e-04, mstar=18.619, mbh=0.052
Resolution TNG50-3: sfr=28.858, bhmdot=2.175e-04, mstar=16.387, mbh=0.069


In [13]:
loaded_data = np.load("subhalo_data.npz")
real_data = loaded_data["data"]
# fields = 'sfr', 'bhmdot', 'mass_stars', 'mass_bhs', 'mean_B'

for j, res in enumerate(["TNG50-1", "TNG50-2", "TNG50-3"]):
    print(f"\nSummary statistics for {res}:")
    valid_mask = real_data[:, j, 0] != -1
    valid_sfrs = real_data[valid_mask, j, 0]
    valid_mstars = real_data[valid_mask, j, 2]
    valid_b_means = real_data[valid_mask, j, 4]

    print(f"  Number of valid subhalos: {len(valid_sfrs)}")
    if len(valid_sfrs) > 0:
        print(f"  Mean Stellar Mass: {np.mean(valid_mstars):.3e} Msun")
        print(f"  Mean SFR: {np.mean(valid_sfrs):.3f} Msun/yr")
        print(f"  Mean B (Mean): {np.mean(valid_b_means):.3e} G")


Summary statistics for TNG50-1:
  Number of valid subhalos: 100
  Mean Stellar Mass: 1.050e+11 Msun
  Mean SFR: 57.272 Msun/yr
  Mean B (Mean): 2.278e-05 G

Summary statistics for TNG50-2:
  Number of valid subhalos: 100
  Mean Stellar Mass: 6.935e+10 Msun
  Mean SFR: 28.891 Msun/yr
  Mean B (Mean): 1.552e-05 G

Summary statistics for TNG50-3:
  Number of valid subhalos: 100
  Mean Stellar Mass: 5.855e+10 Msun
  Mean SFR: 24.316 Msun/yr
  Mean B (Mean): 1.483e-05 G


In [15]:
# 0: sfr, 1: bhmdot, 2: mass_stars, 3: mass_bhs, 4: total_halo_mass, 5: B_mean, 6: B_median
for res_idx in range(3):
    b_mean = real_data[:, res_idx, 4]
    sfrs = real_data[:, res_idx, 0].copy()
    bhmdots = real_data[:, res_idx, 1].copy()

    N = len(b_mean)
    dof = N - 2

    # Handle zeros in log space by substituting with 0.1 * min non-zero value
    mask_sfrs = sfrs > 0
    mask_bhmdots = bhmdots > 0
    num_zeros_sfrs = np.sum(~mask_sfrs)
    num_zeros_bhmdots = np.sum(~mask_bhmdots)

    if np.sum(mask_sfrs) > 0:
        min_for_zeros_sfrs = 0.1 * np.min(sfrs[mask_sfrs])
    else:
        min_for_zeros_sfrs = 1e-4

    if np.sum(mask_bhmdots) > 0:
        min_for_zeros_bhmdots = 0.1 * np.min(bhmdots[mask_bhmdots])
    else:
        min_for_zeros_bhmdots = 1e-6

    sfrs[~mask_sfrs] = min_for_zeros_sfrs
    bhmdots[~mask_bhmdots] = min_for_zeros_bhmdots

    # Log quantities
    log_bmean = np.log10(b_mean)
    log_sfrs = np.log10(sfrs)
    log_bhmdots = np.log10(bhmdots)

    # Spearman and Pearson calculations
    rho_sfr, p_rho_sfr = stats.spearmanr(b_mean, sfrs)
    r_sfr, p_r_sfr = stats.pearsonr(log_bmean, log_sfrs)

    rho_bhmdot, p_rho_bhmdot = stats.spearmanr(b_mean, bhmdots)
    r_bhmdot, p_r_bhmdot = stats.pearsonr(log_bmean, log_bhmdots)

    rho_sfr_bhm, p_rho_sfr_bhm = stats.spearmanr(sfrs, bhmdots)
    r_sfr_bhm, p_r_sfr_bhm = stats.pearsonr(log_sfrs, log_bhmdots)

    # 5-Fold Cross Validation
    num_folds = 5
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

    # 1. sfrXb-field: Predicting log B-field from log SFR
    bmean_preds_sfr = np.zeros_like(log_bmean)
    for train_ind, val_ind in kf.split(log_sfrs):
        X_train, X_val = log_sfrs[train_ind].reshape(-1, 1), log_sfrs[val_ind].reshape(-1, 1)
        y_train = log_bmean[train_ind]
        model = LinearRegression()
        model.fit(X_train, y_train)
        bmean_preds_sfr[val_ind] = model.predict(X_val)
    rmse_sfr_b = np.sqrt(mean_squared_error(log_bmean, bmean_preds_sfr))
    r2_sfr_b = r2_score(log_bmean, bmean_preds_sfr)

    # 2. b-fieldXbhmdot: Predicting log B-field from log BHMDOT
    bmean_preds_bhmdots = np.zeros_like(log_bmean)
    for train_ind, val_ind in kf.split(log_bhmdots):
        X_train, X_val = log_bhmdots[train_ind].reshape(-1, 1), log_bhmdots[val_ind].reshape(-1, 1)
        y_train = log_bmean[train_ind]
        model = LinearRegression()
        model.fit(X_train, y_train)
        bmean_preds_bhmdots[val_ind] = model.predict(X_val)
    rmse_b_bhmdot = np.sqrt(mean_squared_error(log_bmean, bmean_preds_bhmdots))
    r2_b_bhmdot = r2_score(log_bmean, bmean_preds_bhmdots)

    # 3. sfrXbhmdot: Predicting log BHMDOT from log SFR
    bhmdot_preds_sfr = np.zeros_like(log_bhmdots)
    for train_ind, val_ind in kf.split(log_sfrs):
        X_train, X_val = log_sfrs[train_ind].reshape(-1, 1), log_sfrs[val_ind].reshape(-1, 1)
        y_train = log_bhmdots[train_ind]
        model = LinearRegression()
        model.fit(X_train, y_train)
        bhmdot_preds_sfr[val_ind] = model.predict(X_val)
    rmse_sfr_bhmdot = np.sqrt(mean_squared_error(log_bhmdots, bhmdot_preds_sfr))
    r2_sfr_bhmdot = r2_score(log_bhmdots, bhmdot_preds_sfr)

    print(f"\n{'='*20} TNG50-{res_idx+1} {'='*20}")
    print(f"N, d.o.f.: {N}, {dof}")
    print(f"Zeros (SFR, BHMDOT): {num_zeros_sfrs}, {num_zeros_bhmdots}")
    print(f"SFR vs BHMDOT:   Spearman rho = {rho_sfr_bhm:.4f} (p={p_rho_sfr_bhm:.2e}) | Pearson r = {r_sfr_bhm:.4f} (p={p_r_sfr_bhm:.2e})")
    print(f"SFR vs B-field:  Spearman rho = {rho_sfr:.4f} (p={p_rho_sfr:.2e}) | Pearson r = {r_sfr:.4f} (p={p_r_sfr:.2e})")
    print(f"BHMDOT vs B-field: Spearman rho = {rho_bhmdot:.4f} (p={p_rho_bhmdot:.2e}) | Pearson r = {r_bhmdot:.4f} (p={p_r_bhmdot:.2e})")
    print(f"CV sfrXb-field (SFR -> B-field): RMSE={rmse_sfr_b:.4f}, R2={r2_sfr_b:.4f}")
    print(f"CV b-fieldXbhmdot (BHMDOT -> B-field): RMSE={rmse_b_bhmdot:.4f}, R2={r2_b_bhmdot:.4f}")
    print(f"CV sfrXbhmdot (SFR -> BHMDOT): RMSE={rmse_sfr_bhmdot:.4f}, R2={r2_sfr_bhmdot:.4f}")


==================== TNG50-1 ====================
N, d.o.f.: 100, 98
Zeros (SFR, BHMDOT): 0, 4
SFR vs BHMDOT:   Spearman rho = 0.3312 (p=7.64e-04) | Pearson r = 0.3695 (p=1.54e-04)
SFR vs B-field:  Spearman rho = 0.4259 (p=9.98e-06) | Pearson r = 0.4904 (p=2.23e-07)
BHMDOT vs B-field: Spearman rho = 0.4999 (p=1.19e-07) | Pearson r = 0.4245 (p=1.07e-05)
CV sfrXb-field (SFR -> B-field): RMSE=0.3975, R2=0.1958
CV b-fieldXbhmdot (BHMDOT -> B-field): RMSE=0.4080, R2=0.1527
CV sfrXbhmdot (SFR -> BHMDOT): RMSE=0.8657, R2=0.1139

==================== TNG50-2 ====================
N, d.o.f.: 100, 98
Zeros (SFR, BHMDOT): 1, 2
SFR vs BHMDOT:   Spearman rho = 0.5572 (p=1.73e-09) | Pearson r = 0.5953 (p=6.47e-11)
SFR vs B-field:  Spearman rho = 0.4919 (p=2.02e-07) | Pearson r = 0.3821 (p=8.74e-05)
BHMDOT vs B-field: Spearman rho = 0.5886 (p=1.19e-10) | Pearson r = 0.4956 (p=1.58e-07)
CV sfrXb-field (SFR -> B-field): RMSE=0.4912, R2=0.0273
CV b-fieldXbhmdot (BHMDOT -> B-field): RMSE=0.4540, R2=0.168